In [1]:
# Cell 1: Install PySpark & Initialize SparkSession
!pip install pyspark -q

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Anteraja_Bottleneck_Analysis") \
    .getOrCreate()

print(f"PySpark Version: {spark.version}")

PySpark Version: 4.0.4


In [2]:
# Cell 2: Generate Sample Dataset scan_events dengan Edge Cases
from datetime import datetime, timedelta
import random

# Seed untuk reproducilbility
random.seed(42)

hubs = ['HUB_CGK', 'HUB_SUB', 'HUB_BDG', 'HUB_MKS', 'HUB_MES', 'HUB_JOG']
couriers = ['Satria 1', 'Satria 2', 'Satria 3']

data = []
base_time = datetime(2026, 10, 1, 8, 0, 0)

# Generate 1000 paket dengan variasi skenario
for i in range(1, 1001):
    pkg_id = f"PKG{i:05d}"
    hub = random.choice(hubs)

    # Tentukan baseline dwell time per hub (HUB_MKS & HUB_SUB dibuat lebih lama/bottleneck)
    if hub == 'HUB_MKS':
        dwell_hours = random.uniform(8.0, 14.0)
    elif hub == 'HUB_SUB':
        dwell_hours = random.uniform(6.0, 11.0)
    elif hub == 'HUB_BDG':
        dwell_hours = random.uniform(5.0, 9.0)
    else:
        dwell_hours = random.uniform(1.0, 4.0)

    arr_time = base_time + timedelta(hours=random.uniform(0, 48))
    dep_time = arr_time + timedelta(hours=dwell_hours)

    # Skenario 1: Normal Valid Pair (90% data)
    if i <= 900:
        data.append((hub, pkg_id, 'ARRIVAL', arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub, pkg_id, 'DEPARTURE', dep_time.strftime('%Y-%m-%d %H:%M:%S')))
    # Skenario 2: Missing DEPARTURE (5% data)
    elif i <= 950:
        data.append((hub, pkg_id, 'ARRIVAL', arr_time.strftime('%Y-%m-%d %H:%M:%S')))
    # Skenario 3: Duplicate Scan (3% data)
    elif i <= 980:
        data.append((hub, pkg_id, 'ARRIVAL', arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub, pkg_id, 'ARRIVAL', arr_time.strftime('%Y-%m-%d %H:%M:%S'))) # Duplicate
        data.append((hub, pkg_id, 'DEPARTURE', dep_time.strftime('%Y-%m-%d %H:%M:%S')))
    # Skenario 4: DEPARTURE sebelum ARRIVAL / Invalid Timestamp (2% data)
    else:
        invalid_dep = arr_time - timedelta(hours=2) # Dep lebih dulu dari Arr
        data.append((hub, pkg_id, 'ARRIVAL', arr_time.strftime('%Y-%m-%d %H:%M:%S')))
        data.append((hub, pkg_id, 'DEPARTURE', invalid_dep.strftime('%Y-%m-%d %H:%M:%S')))

# Buat DataFrame PySpark
schema = ["hub_id", "package_id", "event_type", "timestamp"]
df_raw = spark.createDataFrame(data, schema)
df_raw = df_raw.withColumn("timestamp", F.to_timestamp("timestamp"))

print(f"Total Raw Scan Events: {df_raw.count()}")
df_raw.show(10, truncate=False)

Total Raw Scan Events: 1980
+-------+----------+----------+-------------------+
|hub_id |package_id|event_type|timestamp          |
+-------+----------+----------+-------------------+
|HUB_JOG|PKG00001  |ARRIVAL   |2026-10-02 19:35:39|
|HUB_JOG|PKG00001  |DEPARTURE |2026-10-02 20:55:42|
|HUB_SUB|PKG00002  |ARRIVAL   |2026-10-02 19:21:02|
|HUB_SUB|PKG00002  |DEPARTURE |2026-10-03 02:28:00|
|HUB_JOG|PKG00003  |ARRIVAL   |2026-10-02 10:10:39|
|HUB_JOG|PKG00003  |DEPARTURE |2026-10-02 13:23:58|
|HUB_MES|PKG00004  |ARRIVAL   |2026-10-01 09:25:48|
|HUB_MES|PKG00004  |DEPARTURE |2026-10-01 11:41:45|
|HUB_SUB|PKG00005  |ARRIVAL   |2026-10-02 12:53:48|
|HUB_SUB|PKG00005  |DEPARTURE |2026-10-02 20:03:36|
+-------+----------+----------+-------------------+
only showing top 10 rows


In [3]:
# Cell 3: Data Quality Check & Filtering Valid Scan Pairs

print("=== 1. SCHEMA DATASET ===")
df_raw.printSchema()

print("\n=== 2. AUDIT SCAN DUPLIKAT ===")
df_dedup = df_raw.dropDuplicates(["hub_id", "package_id", "event_type", "timestamp"])
duplicate_count = df_raw.count() - df_dedup.count()
print(f"Jumlah scan duplikat dibuang: {duplicate_count}")

# Pivot data untuk memisahkan ARRIVAL dan DEPARTURE per package per hub
df_pivoted = df_dedup.groupBy("hub_id", "package_id").agg(
    F.min(F.when(F.col("event_type") == "ARRIVAL", F.col("timestamp"))).alias("arrival_time"),
    F.max(F.when(F.col("event_type") == "DEPARTURE", F.col("timestamp"))).alias("departure_time"),
    F.sum(F.when(F.col("event_type") == "ARRIVAL", 1).otherwise(0)).alias("arrival_count"),
    F.sum(F.when(F.col("event_type") == "DEPARTURE", 1).otherwise(0)).alias("departure_count")
)

print("\n=== 3. AUDIT PAIR QUALITY ===")
total_pairs = df_pivoted.count()
missing_arrival = df_pivoted.filter(F.col("arrival_time").isNull()).count()
missing_departure = df_pivoted.filter(F.col("departure_time").isNull()).count()
invalid_sequence = df_pivoted.filter(F.col("departure_time") < F.col("arrival_time")).count()

print(f"Total Pasangan Paket x Hub : {total_pairs}")
print(f"Missing ARRIVAL            : {missing_arrival}")
print(f"Missing DEPARTURE          : {missing_departure}")
print(f"Invalid Sequence (Dep < Arr): {invalid_sequence}")

# Filter hanya pasangan scan yang VALID
df_valid_pairs = df_pivoted.filter(
    F.col("arrival_time").isNotNull() &
    F.col("departure_time").isNotNull() &
    (F.col("departure_time") >= F.col("arrival_time"))
)

print(f"\nJumlah Pasangan Valid Siap Analisis: {df_valid_pairs.count()}")

=== 1. SCHEMA DATASET ===
root
 |-- hub_id: string (nullable = true)
 |-- package_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)


=== 2. AUDIT SCAN DUPLIKAT ===
Jumlah scan duplikat dibuang: 30

=== 3. AUDIT PAIR QUALITY ===
Total Pasangan Paket x Hub : 1000
Missing ARRIVAL            : 0
Missing DEPARTURE          : 50
Invalid Sequence (Dep < Arr): 20

Jumlah Pasangan Valid Siap Analisis: 930


In [4]:
# Cell 4: Hitung Dwell Time (Jam)
df_dwell = df_valid_pairs.withColumn(
    "dwell_hours",
    (F.col("departure_time").cast("long") - F.col("arrival_time").cast("long")) / 3600.0
)

df_dwell.select("hub_id", "package_id", "arrival_time", "departure_time", "dwell_hours").show(10, truncate=False)

+-------+----------+-------------------+-------------------+------------------+
|hub_id |package_id|arrival_time       |departure_time     |dwell_hours       |
+-------+----------+-------------------+-------------------+------------------+
|HUB_MKS|PKG00049  |2026-10-03 04:35:48|2026-10-03 12:42:48|8.116666666666667 |
|HUB_MES|PKG00055  |2026-10-02 04:18:38|2026-10-02 07:55:40|3.6172222222222223|
|HUB_CGK|PKG00076  |2026-10-03 07:57:56|2026-10-03 09:07:02|1.1516666666666666|
|HUB_MES|PKG00182  |2026-10-01 08:16:58|2026-10-01 10:25:26|2.141111111111111 |
|HUB_BDG|PKG00235  |2026-10-02 18:37:17|2026-10-03 00:32:37|5.9222222222222225|
|HUB_JOG|PKG00325  |2026-10-02 05:17:47|2026-10-02 08:54:51|3.617777777777778 |
|HUB_SUB|PKG00061  |2026-10-01 08:20:42|2026-10-01 17:17:14|8.942222222222222 |
|HUB_JOG|PKG00001  |2026-10-02 19:35:39|2026-10-02 20:55:42|1.3341666666666667|
|HUB_JOG|PKG00137  |2026-10-02 04:56:48|2026-10-02 08:18:22|3.3594444444444442|
|HUB_CGK|PKG00426  |2026-10-02 15:34:33|

In [5]:
# Cell 5: Aggregate per Hub & Export CSV
import pandas as pd

# Hitung aggregate
df_hub_summary = df_dwell.groupBy("hub_id").agg(
    F.count("package_id").alias("package_count"),
    F.round(F.mean("dwell_hours"), 2).alias("avg_dwell_hours"),
    F.round(F.expr("percentile_approx(dwell_hours, 0.5)"), 2).alias("median_dwell_hours")
).orderBy(F.col("avg_dwell_hours").desc())

print("=== REKAPITULASI BOTTLE NECK HUB ===")
df_hub_summary.show(truncate=False)

# Konversi ke Pandas & simpan CSV
df_pandas = df_hub_summary.toPandas()
df_pandas.to_csv('top_bottleneck_hubs.csv', index=False)
print("File top_bottleneck_hubs.csv berhasil dibuat!")

=== REKAPITULASI BOTTLE NECK HUB ===
+-------+-------------+---------------+------------------+
|hub_id |package_count|avg_dwell_hours|median_dwell_hours|
+-------+-------------+---------------+------------------+
|HUB_MKS|164          |10.97          |11.01             |
|HUB_SUB|155          |8.39           |8.29              |
|HUB_BDG|152          |6.84           |6.82              |
|HUB_JOG|165          |2.53           |2.51              |
|HUB_MES|157          |2.48           |2.57              |
|HUB_CGK|137          |2.43           |2.3               |
+-------+-------------+---------------+------------------+

File top_bottleneck_hubs.csv berhasil dibuat!
